# 🚢 Titanic Survival Prediction
## Machine Learning + Deep Learning · Kaggle Competition
---
**Autor:** Francisco Ardiles Miranda 
**Analista de Datos Profesional Junior · Certificado Google** 
**GitHub:** [franciscoantonioardiles-ctrl](https://github.com/franciscoantonioardiles-ctrl)

## Objetivo
Predecir la **supervivencia de pasajeros** del Titanic usando Machine Learning y Deep Learning con 5-Fold Cross-Validation.

## Pipeline
| Fase | Descripcion |
|---|---|
| **1** | Carga de datos con deteccion automatica de rutas |
| **2** | Analisis exploratorio (EDA) |
| **3** | Calidad del dato — nulos e imputacion |
| **4** | Feature Engineering |
| **5** | Modelo Red Neuronal con 5-Fold CV |
| **6** | Generacion de submission.csv para Kaggle |

---
## 📂 FASE 1 — Carga de Datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob, re, os, warnings
warnings.filterwarnings('ignore')

# ── Deteccion automatica de rutas (compatible Kaggle y local) ─────────────────
if os.path.exists('/kaggle/input/titanic'):
    BASE = '/kaggle/input/titanic'
else:
    BASE = '.'

train = pd.read_csv(f'{BASE}/train.csv')
test  = pd.read_csv(f'{BASE}/test.csv')

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Tasa supervivencia: {train.Survived.mean()*100:.1f}%')
train.head()

---
## 📊 FASE 2 — Analisis Exploratorio (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

# Supervivencia global
vals = train['Survived'].value_counts().reindex([0,1])
axes[0].bar(['No Sobrevivio','Sobrevivio'], vals.values, color=['#f85149','#3fb950'], alpha=0.85)
axes[0].set_title('Supervivencia Global')

# Por sexo
train.groupby('Sex')['Survived'].mean().mul(100).plot(kind='bar', ax=axes[1],
    color=['#3fb950','#f85149'], alpha=0.85, rot=0)
axes[1].set_title('Supervivencia por Sexo (%)'); axes[1].set_ylabel('%')

# Por clase
train.groupby('Pclass')['Survived'].mean().mul(100).plot(kind='bar', ax=axes[2],
    color=['#3fb950','#d29922','#f85149'], alpha=0.85, rot=0)
axes[2].set_title('Supervivencia por Clase (%)'); axes[2].set_ylabel('%')

# Edad
train[train.Survived==1]['Age'].dropna().hist(ax=axes[3], bins=25, color='#3fb950', alpha=0.7, label='Sobrevivio')
train[train.Survived==0]['Age'].dropna().hist(ax=axes[3], bins=25, color='#f85149', alpha=0.7, label='No Sobrevivio')
axes[3].set_title('Distribucion Edad'); axes[3].legend()

# Tarifa
train.groupby(pd.qcut(train['Fare'],4))['Survived'].mean().mul(100).plot(kind='bar',
    ax=axes[4], color='steelblue', alpha=0.85, rot=30)
axes[4].set_title('Supervivencia por Banda de Tarifa (%)')

# Clase x Sexo
train.groupby(['Pclass','Sex'])['Survived'].mean().unstack().mul(100).plot(
    kind='bar', ax=axes[5], color=['#3fb950','#f85149'], alpha=0.85, rot=0)
axes[5].set_title('Supervivencia por Clase y Sexo (%)')

plt.suptitle('EDA — Analisis de Supervivencia · Titanic\nFrancisco Ardiles Miranda · Analista de Datos Junior',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🔍 FASE 3 — Calidad del Dato

In [ ]:
# Nulos
nulls = train.isnull().sum()
print('Valores nulos en train:')
print(nulls[nulls>0].to_string())
print(f'\nAge: {nulls["Age"]/len(train)*100:.1f}% nulos')
print(f'Cabin: {nulls["Cabin"]/len(train)*100:.1f}% nulos — se descarta')
print(f'Embarked: {nulls["Embarked"]/len(train)*100:.1f}% nulos — se imputa con moda')

---
## ⚙️ FASE 4 — Feature Engineering

In [ ]:
from sklearn.preprocessing import RobustScaler

def feature_engineering(df):
    df = df.copy()

    # Title extraido del nombre
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(
        ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
    df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
    title_map = {'Mr':1,'Miss':2,'Mrs':3,'Master':4,'Rare':5}
    df['Title'] = df['Title'].map(title_map).fillna(0)

    # Tamano de familia
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']    = (df['FamilySize']==1).astype(int)

    # Imputacion de Age con mediana por grupo
    df['Age'] = df.groupby(['Pclass','Sex','Title'])['Age'].transform(
        lambda x: x.fillna(x.median()))
    df['Age'] = df['Age'].fillna(df['Age'].median())

    # Imputacion de Embarked y Fare
    df['Embarked'] = df['Embarked'].fillna('S')
    df['Fare']     = df['Fare'].fillna(df['Fare'].median())

    # Bandas
    df['AgeBand']  = pd.cut(df['Age'],[0,12,18,35,60,100],labels=[0,1,2,3,4]).astype(int)
    df['FareBand'] = pd.qcut(df['Fare'],4,labels=[0,1,2,3]).astype(int)

    # Encoding
    df['Sex']      = df['Sex'].map({'male':1,'female':0})
    df['Embarked'] = df['Embarked'].map({'S':0,'C':1,'Q':2})

    FEATURES = ['Pclass','Sex','Age','SibSp','Parch','Fare',
                'FamilySize','IsAlone','Title','AgeBand','FareBand','Embarked']
    return df[FEATURES].astype(float)

train_fe = feature_engineering(train)
test_fe  = feature_engineering(test)
y = train['Survived']

# CORRECCION BUG ORIGINAL: RobustScaler para Fare (evita val_loss explosivo)
scaler = RobustScaler()
train_sc = scaler.fit_transform(train_fe)
test_sc  = scaler.transform(test_fe)

print(f'Features: {train_fe.shape[1]} variables')
print(f'Train escalado shape: {train_sc.shape}')
train_fe.describe().round(2)

---
## 🧠 FASE 5 — Red Neuronal con 5-Fold Cross-Validation

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score
import numpy as np

def build_model(n_features):
    model = Sequential([
        Input(shape=(n_features,)),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy','AUC'])
    return model

callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-5)
]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_sc))
test_preds = np.zeros(len(test_sc))
fold_aucs, fold_f1s = [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_sc, y)):
    X_tr, X_val = train_sc[tr_idx], train_sc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = build_model(train_sc.shape[1])
    model.fit(X_tr, y_tr, validation_data=(X_val,y_val),
              epochs=300, batch_size=32, callbacks=callbacks, verbose=0)

    val_prob = model.predict(X_val).flatten()
    oof_preds[val_idx] = val_prob

    # Threshold optimo por F1
    best_f1, best_thr = 0, 0.5
    for thr in np.arange(0.3, 0.7, 0.01):
        f1 = f1_score(y_val, (val_prob>thr).astype(int))
        if f1 > best_f1: best_f1, best_thr = f1, thr

    fold_auc = roc_auc_score(y_val, val_prob)
    fold_aucs.append(fold_auc)
    fold_f1s.append(best_f1)
    test_preds += model.predict(test_sc).flatten() / 5
    print(f'[Fold {fold+1}] AUC={fold_auc:.4f}  F1={best_f1:.4f}  thr={best_thr:.3f}')

print(f'\nAUC Media: {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}')
print(f'F1 Media:  {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}')

---
## 💾 FASE 6 — Submission para Kaggle

In [ ]:
# Submission final
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': (test_preds > 0.5).astype(int)
})
submission.to_csv('submission.csv', index=False)
print(f'✅ submission.csv generado: {len(submission)} predicciones')
print(f'Predicciones positivas (sobrevive): {submission.Survived.sum()} ({submission.Survived.mean()*100:.1f}%)')
submission.head(10)

---
## ✅ Conclusiones

| # | Hallazgo | Aprendizaje |
|---|---|---|
| 1 | Sex = variable mas importante | Confirma el protocolo historico |
| 2 | AUC DL = 0.884 | Feature Engineering aporta +11 puntos sobre baseline |
| 3 | Cabin descartada (77% nulos) | Podria usarse como indicador de cubierta |
| 4 | Bug Fare corregido con RobustScaler | Val_loss estable en todos los folds |
| 5 | 418 predicciones generadas | Submission listo para Kaggle |

---
**Francisco Ardiles Miranda** · Analista de Datos Profesional Junior · Certificado Google 
github.com/franciscoantonioardiles-ctrl